# Proposal rankings within the Chilean left and right

**Inputs:** `data/processed/Chile_pairs.parquet`, participant age decisions, and the Chilean metadata and proposal labels in `data/raw/`. Notebook 02 regenerates the supplied Chilean pairs, and notebook 01 regenerates the age decisions. This notebook runs independently in a fresh kernel from the included files.

**Outputs:** group rankings and the complete repetition-level ratings in `outputs/tables/proposal_rankings/`, plus `data/processed/proposal_rankings.csv`, used by notebooks 09, 13 and 16. Run all cells to recompute the rankings. The 200 full passes over observed comparisons take longer than plotting. Repetition count and random seed are exposed below for exploration.

This notebook recomputes proposal rankings from the age-screened Chilean pairwise choices. It follows the historical analysis: 100 complete random permutations of the observed choices within each ideological group, TrueSkill updates in each order, and rankings derived from the mean conservative score `mu - 3*sigma`. These repetitions vary update order; they do **not** resample with replacement and are not bootstrap confidence intervals.

Participants are classified as left for reported positions 0–4 and right for positions 6–10. The central position and nonresponses have no side classification. Repeated metadata never duplicate comparisons: one unambiguous binary side is assigned per ID, and IDs with both left and right responses are excluded. Previously excluded ages cannot enter because the input pairs and IDs are checked against the participant age decisions. Participants without a reported age remain eligible if their side is known.

The original prior is `mu = number_of_proposals / 2`, `sigma = number_of_proposals / 6`. The TrueSkill environment retains its original defaults: beta `25/6`, tau `25/300`, draw probability `0.1`, and the internal normal-CDF approximation. The prior is not used to rescale beta or tau. No draws occur in the retained pairwise input.

A small numerical module accelerates the repeated updates. Before the full run, the notebook compares every updated state with the original `trueskill.rate_1vs1` on 10,000 real comparisons and 10,000 synthetic comparisons. It must agree within an absolute tolerance of `1e-7` for mu, sigma, and conservative skill. The original CDF approximation's copyright and BSD license accompany `src/trueskill_fast.py`.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import trueskill
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "data/raw").is_dir(), "Open from the project root or notebooks directory."

sys.path.insert(0, str(ROOT / "src"))
from trueskill_fast import rate_sequence

TABLES = ROOT / "outputs/tables/proposal_rankings"
TABLES.mkdir(parents=True, exist_ok=True)
# Keep the original TrueSkill environment, independent of the proposal-count prior.
ENV = trueskill.TrueSkill()
assert (
    ENV.backend is None
    and np.isclose(ENV.beta, 25 / 6, rtol=0, atol=1e-15)
    and np.isclose(ENV.tau, 25 / 300, rtol=0, atol=1e-15)
    and ENV.draw_probability == 0.1
)
DRAW_MARGIN = trueskill.calc_draw_margin(ENV.draw_probability, 2, ENV)
REPETITIONS = 100
PERMUTATION_SEED = 42
TOLERANCE = 1e-7

pairs = pd.read_parquet(ROOT / "data" / "processed" / "Chile_pairs.parquet")
metadata = pd.read_csv(ROOT / "data" / "raw" / "chile_metadata.csv", dtype={"uuid": str})
age_decisions = pd.read_csv(
    ROOT / "data" / "processed" / "participant_age_decisions.csv", dtype={"uuid": str}
)
labels = pd.read_csv(ROOT / "data" / "raw" / "chile_labels.tsv", sep="\t")
assert labels.id.is_unique
assert pairs.id.is_unique
assert ((pairs.selected == pairs.option_a) | (pairs.selected == pairs.option_b)).all()
assert (pairs.option_a != pairs.option_b).all()
chile_decisions = age_decisions[age_decisions.country.str.casefold() == "chile"].set_index(
    "uuid"
)
assert not chile_decisions.index.duplicated().any()
assert set(pairs.uuid).issubset(chile_decisions.index)
assert chile_decisions.loc[pairs.uuid.unique(), "include"].all()
expected_ids = set(
    chile_decisions.index[chile_decisions.include & chile_decisions.in_original_main_fit]
)
assert set(pairs.uuid) == expected_ids
proposal_ids = sorted(set(pairs.option_a) | set(pairs.option_b))
assert set(proposal_ids).issubset(set(labels.id))


## Assign one binary side per participant

The join below cannot multiply choice rows. Differences between two positions on the same side are allowed; conflicting left/right responses exclude the participant. No other demographic variable selects the ranking sample.

In [2]:
metadata = metadata[metadata.uuid.isin(pairs.uuid)].copy()
metadata["politica"] = pd.to_numeric(metadata.politica, errors="coerce")
side_rows = metadata.loc[
    metadata.politica.isin(list(range(5)) + list(range(6, 11))), ["uuid", "politica"]
].copy()
side_rows["side"] = np.where(side_rows.politica < 5, "Left", "Right")
side_rows = side_rows[["uuid", "side"]].drop_duplicates()
conflicting_ids = set(side_rows.loc[side_rows.duplicated("uuid", keep=False), "uuid"])
sides = side_rows[~side_rows.uuid.isin(conflicting_ids)].copy()
assert sides.uuid.is_unique
# Each participant has at most one side, so the join cannot duplicate a comparison.
ranked_pairs = pairs.merge(sides, on="uuid", how="inner", validate="many_to_one")
assert ranked_pairs.id.is_unique
assert not ranked_pairs.uuid.isin(conflicting_ids).any()
assert set(ranked_pairs.uuid).issubset(expected_ids)

exclusions = pd.DataFrame(
    {"uuid": sorted(conflicting_ids), "reason": "conflicting_left_right_responses"}
)
display(exclusions)
exclusions.to_csv(TABLES / "excluded_side_conflicts.csv", index=False)
display(sides)
sides.to_csv(TABLES / "participant_sides.csv", index=False)
sample_summary = (
    ranked_pairs.groupby("side")
    .agg(participants=("uuid", "nunique"), comparisons=("id", "size"))
    .reset_index()
)
sample_summary["proposals"] = [
    len(set(g.option_a) | set(g.option_b)) for _, g in ranked_pairs.groupby("side")
]
display(sample_summary)
sample_summary.to_csv(TABLES / "ranking_sample.csv", index=False)
sample_summary.to_latex(TABLES / "ranking_sample.tex", index=False)


,uuid,reason
0,1781ae79-cc01-416a-8437-cda5501cd59d,conflicting_left_right_responses
1,18104105-f9e5-479a-a509-42285e67b9dc,conflicting_left_right_responses
2,1bdee467-b056-45dc-8c1d-6c47b591c89c,conflicting_left_right_responses
3,2a5b45e9-db43-4d92-bfa6-e471f95d8fe1,conflicting_left_right_responses
4,34769db1-9d8a-4dc7-9b22-a806c810f03e,conflicting_left_right_responses
5,4b0b02d3-af8f-4f26-a6f9-7713d46d0e23,conflicting_left_right_responses
6,59a4ff59-114d-4c79-815e-8fae385be646,conflicting_left_right_responses
7,5b246b2a-c2ca-4f9e-9f56-d283692aa3da,conflicting_left_right_responses
8,5dd947b0-d656-47d5-acd8-228aac6a3e41,conflicting_left_right_responses
9,6f53593e-cc8a-4779-accc-a10f37d6a1de,conflicting_left_right_responses


,uuid,side
1,8bd42737-b140-4db8-af4c-59839922129a,Left
4,7d2e034a-3270-4429-aeac-418c46bf0d7d,Right
5,c1b97bb6-03d2-4027-8d54-0dd64a927a6c,Left
8,883528e9-0599-4d30-9f4c-1daf9cf74082,Right
10,7fd1210c-b1c0-4bdf-b190-7bfa10fe51d2,Left
...,...,...
44281,759b16f2-f527-4f04-85ba-682fd1af8a89,Left
44286,2e8f5550-d794-411c-b36e-d277e88f5d30,Right
44288,3ddc9eae-ba7b-4dab-8e43-5c188daf9692,Left
44294,b7132b2c-5ca4-44ac-b4e7-07e43d4b7547,Right


,side,participants,comparisons,proposals
0,Left,12473,870892,90
1,Right,9570,700563,90


## Verify the accelerated updates against the original implementation

Both checks start with the same priors, default environment, and ordered comparisons. The reported maximum error includes every winner/loser mean and standard deviation after each of the 10,000 updates, not only the final ratings. Failure of the tolerance stops execution before any complete ranking run.

In [3]:
proposal_index = {identifier: index for index, identifier in enumerate(proposal_ids)}
real = pairs.iloc[:10_000]
real_winners = real.selected.map(proposal_index).to_numpy(dtype=np.int64)
# The unselected member of each pair is the loser. Map source IDs to array positions.
real_losers = (
    pd.Series(np.where(real.selected == real.option_a, real.option_b, real.option_a))
    .map(proposal_index)
    .to_numpy(dtype=np.int64)
)
rng_test = np.random.RandomState(2026)
synthetic_winners = rng_test.randint(0, len(proposal_ids), 10_000).astype(np.int64)
# A nonzero modular offset guarantees distinct winners and losers in this test.
synthetic_losers = (synthetic_winners + rng_test.randint(1, len(proposal_ids), 10_000)) % len(
    proposal_ids
)
proof_rows = []
for case, winners, losers in [
    ("real_10000", real_winners, real_losers),
    ("synthetic_10000", synthetic_winners, synthetic_losers),
]:
    n_options = len(proposal_ids)
    initial_mu, initial_sigma = n_options / 2, n_options / 6
    mu, sigma, trace = rate_sequence(
        winners,
        losers,
        n_options,
        DRAW_MARGIN,
        ENV.beta,
        ENV.tau,
        initial_mu,
        initial_sigma,
        True,
    )
    reference = [trueskill.Rating(initial_mu, initial_sigma) for _ in proposal_ids]
    # Trace stores winner mean/SD and loser mean/SD after every update.
    reference_trace = np.empty_like(trace)
    for i, (winner, loser) in enumerate(zip(winners, losers)):
        reference[winner], reference[loser] = trueskill.rate_1vs1(
            reference[winner], reference[loser], env=ENV
        )
        reference_trace[i] = [
            reference[winner].mu,
            reference[winner].sigma,
            reference[loser].mu,
            reference[loser].sigma,
        ]
    reference_mu = np.array([rating.mu for rating in reference])
    reference_sigma = np.array([rating.sigma for rating in reference])
    errors = {
        "mu_error": float(np.max(np.abs(mu - reference_mu))),
        "sigma_error": float(np.max(np.abs(sigma - reference_sigma))),
        "skill_error": float(
            np.max(np.abs((mu - 3 * sigma) - (reference_mu - 3 * reference_sigma)))
        ),
        "per_update_state_error": float(np.max(np.abs(trace - reference_trace))),
    }
    assert max(errors.values()) <= TOLERANCE, (case, errors)
    proof_rows.append(
        {"case": case, "comparisons": len(winners), "tolerance": TOLERANCE, **errors}
    )

proof = pd.DataFrame(proof_rows)
display(proof)
proof.to_csv(TABLES / "trueskill_equivalence.csv", index=False)
proof.to_latex(TABLES / "trueskill_equivalence.tex", index=False, float_format="%.12e")


,case,comparisons,tolerance,mu_error,sigma_error,skill_error,per_update_state_error
0,real_10000,10000,1.000000e-07,1.136868e-13,8.881784e-16,1.136868e-13,1.350031e-13
1,synthetic_10000,10000,1.000000e-07,1.065814e-13,1.221245e-15,1.065814e-13,1.136868e-13


## Repeat the rankings over random update orders

A single seeded random stream supplies 100 complete permutations for the left, then 100 for the right. Every comparison appears exactly once per repetition. Priors are reset each time. This varies the order of sequential TrueSkill updates, not which observations enter the analysis.

Each update uses integer array positions rather than proposal IDs. The mapping is rebuilt for each side and restored in the result table. The per-repetition rank orders `mu - 3*sigma`. Later we average these conservative scores and rank their means. Averaging the ranks themselves is a different quantity, retained in a separate column.

Progress is printed after the first repetition and every ten repetitions. The saved repetition-level table lets you inspect variability across update orders and recompute the final mean scores.


In [4]:
rng = np.random.RandomState(PERMUTATION_SEED)
replicate_frames = []
for side in ["Left", "Right"]:
    group = ranked_pairs[ranked_pairs.side == side]
    ids = sorted(set(group.option_a) | set(group.option_b))
    n_options = len(ids)
    index = {identifier: number for number, identifier in enumerate(ids)}
    winners = group.selected.map(index).to_numpy(dtype=np.int64)
    losers = (
        pd.Series(np.where(group.selected == group.option_a, group.option_b, group.option_a))
        .map(index)
        .to_numpy(dtype=np.int64)
    )
    assert len(winners) == len(losers) == len(group)
    assert not np.isnan(winners).any() and not np.isnan(losers).any()
    won = np.bincount(winners, minlength=n_options)
    lost = np.bincount(losers, minlength=n_options)
    for repetition in range(1, REPETITIONS + 1):
        # Every index appears exactly once, unlike a bootstrap sample.
        order = rng.permutation(len(group))
        mu, sigma, _ = rate_sequence(
            winners[order],
            losers[order],
            n_options,
            DRAW_MARGIN,
            ENV.beta,
            ENV.tau,
            n_options / 2,
            n_options / 6,
        )
        result = pd.DataFrame(
            {
                "id": ids,
                "mu": mu,
                "sigma": sigma,
                "skill": mu - 3 * sigma,
                "won": won,
                "lost": lost,
                "score": won - lost,
            }
        )
        assert np.isfinite(result.to_numpy()).all()
        assert (result.sigma > 0).all()
        result = result.sort_values("skill", ascending=False)
        result["rank"] = np.arange(1, n_options + 1)
        result.insert(0, "side", side)
        result.insert(1, "repetition", repetition)
        replicate_frames.append(result)
        if repetition == 1 or repetition % 10 == 0:
            print(
                f"{side}: {repetition}/{REPETITIONS} complete orders, {len(group):,} comparisons per order",
                flush=True,
            )

replicates = pd.concat(replicate_frames, ignore_index=True)
assert replicates.groupby(["side", "id"]).size().eq(REPETITIONS).all()
display(replicates)
replicates.to_csv(TABLES / "trueskill_replicates.csv", index=False)


Left: 1/100 complete orders, 870,892 comparisons per order


Left: 10/100 complete orders, 870,892 comparisons per order


Left: 20/100 complete orders, 870,892 comparisons per order


Left: 30/100 complete orders, 870,892 comparisons per order


Left: 40/100 complete orders, 870,892 comparisons per order


...

Right: 70/100 complete orders, 700,563 comparisons per order


Right: 80/100 complete orders, 700,563 comparisons per order


Right: 90/100 complete orders, 700,563 comparisons per order


Right: 100/100 complete orders, 700,563 comparisons per order


,side,repetition,id,mu,sigma,skill,won,lost,score,rank
0,Left,1,23,52.456752,0.878147,49.822312,22047,4194,17853,1
1,Left,1,6,51.296668,0.893711,48.615535,15795,3707,12088,2
2,Left,1,55,50.098008,0.850932,47.545211,15806,3559,12247,3
3,Left,1,37,50.057407,0.848409,47.512180,15035,4315,10720,4
4,Left,1,85,49.449045,0.843518,46.918492,15157,4338,10819,5
...,...,...,...,...,...,...,...,...,...,...
17995,Right,100,29,39.806530,0.826908,37.325805,3692,11141,-7449,86
17996,Right,100,31,39.362636,0.844921,36.827872,4349,10238,-5889,87
17997,Right,100,37,39.133536,0.841634,36.608635,2600,11650,-9050,88
17998,Right,100,2,38.375275,0.853873,35.813656,2088,12519,-10431,89


## Average conservative scores and construct rank ratios

`Ranking Izquierda` and `Ranking Derecha` are ranks of the mean conservative score, starting at one for the highest-ranked proposal. The two primary ratio columns are left rank divided by right rank and its reciprocal. The older score-ratio and mean-rank-ratio columns are retained for compatibility; they do not replace the primary rank ratios.

In [5]:
group_rankings = {}
for side in ["Left", "Right"]:
    means = (
        replicates[replicates.side == side]
        .groupby("id")[["mu", "sigma", "skill", "won", "lost", "score", "rank"]]
        .mean()
        .reset_index()
    )
    means = means.sort_values("skill", ascending=False)
    means["rank_of_mean_skill"] = np.arange(1, len(means) + 1)
    group_rankings[side] = means
    display(means)
    means.to_csv(TABLES / f"trueskill_{side.lower()}_mean.csv", index=False)
    means.to_latex(
        TABLES / f"trueskill_{side.lower()}_mean.tex", index=False, float_format="%.6f"
    )

left = group_rankings["Left"].rename(
    columns={"rank_of_mean_skill": "Ranking Izquierda", "skill": "skill iz", "rank": "rank iz"}
)
right = group_rankings["Right"].rename(
    columns={"rank_of_mean_skill": "Ranking Derecha", "skill": "skill der", "rank": "rank der"}
)
rankings = left[["id", "Ranking Izquierda", "skill iz", "rank iz"]].merge(
    right[["id", "Ranking Derecha", "skill der", "rank der"]], on="id", validate="one_to_one"
)
rankings = rankings.merge(
    labels[["id", "short_name", "topic_name", "name"]],
    on="id",
    how="left",
    validate="one_to_one",
)
# Smaller left/right ratios identify proposals ranked more highly by the left.
rankings["ranking_derecha_sort"] = rankings["Ranking Derecha"] / rankings["Ranking Izquierda"]
rankings["ranking_izquierda_sort"] = rankings["Ranking Izquierda"] / rankings["Ranking Derecha"]
rankings["ranking_derecha_sort_skill"] = rankings["skill der"] / rankings["skill iz"]
rankings["ranking_izquierda_sort_skill"] = rankings["skill iz"] / rankings["skill der"]
rankings["ranking_derecha_sort_rank"] = rankings["rank der"] / rankings["rank iz"]
rankings["ranking_izquierda_sort_rank"] = rankings["rank iz"] / rankings["rank der"]
assert rankings.id.is_unique
assert set(rankings.id) == set(proposal_ids)
assert rankings[["short_name", "topic_name", "name"]].notna().all().all()
assert np.allclose(rankings.ranking_derecha_sort * rankings.ranking_izquierda_sort, 1)
display(rankings)
rankings.to_csv(ROOT / "data" / "processed" / "proposal_rankings.csv", index=False)

rank_summary = rankings[
    [
        "id",
        "Ranking Izquierda",
        "Ranking Derecha",
        "skill iz",
        "skill der",
        "rank iz",
        "rank der",
        "ranking_izquierda_sort",
        "ranking_derecha_sort",
    ]
].copy()
display(rank_summary)
rank_summary.to_csv(TABLES / "proposal_rankings_summary.csv", index=False)
rank_summary.to_latex(
    TABLES / "proposal_rankings_summary.tex", index=False, float_format="%.6f"
)


,id,mu,sigma,skill,won,lost,score,rank,rank_of_mean_skill
22,23,53.212090,0.873078,50.592856,22047.0,4194.0,17853.0,1.07,1
54,55,51.516970,0.865766,48.919671,15806.0,3559.0,12247.0,2.61,2
5,6,51.298802,0.858146,48.724362,15795.0,3707.0,12088.0,3.12,3
84,85,50.508770,0.843508,47.978246,15157.0,4338.0,10819.0,4.87,4
36,37,50.492929,0.840906,47.970210,15035.0,4315.0,10720.0,5.03,5
...,...,...,...,...,...,...,...,...,...
10,11,38.773907,0.872615,36.156062,3278.0,15782.0,-12504.0,85.80,86
79,80,38.320243,0.884622,35.666375,2919.0,15847.0,-12928.0,86.66,87
48,49,37.252169,0.915034,34.507068,2324.0,16273.0,-13949.0,88.80,88
51,52,37.249936,0.915490,34.503466,2204.0,16526.0,-14322.0,88.71,89


,id,mu,sigma,skill,won,lost,score,rank,rank_of_mean_skill
44,45,50.008120,0.836097,47.499830,12346.0,3762.0,8584.0,3.79,1
63,64,49.832550,0.833497,47.332061,12386.0,3949.0,8437.0,4.71,2
54,55,49.633088,0.828104,47.148776,11930.0,4052.0,7878.0,6.02,3
55,56,49.564216,0.827882,47.080569,12003.0,4169.0,7834.0,6.13,4
20,21,49.471557,0.827050,46.990408,12210.0,4270.0,7940.0,6.95,5
...,...,...,...,...,...,...,...,...,...
28,29,40.541967,0.831246,38.048229,3692.0,11141.0,-7449.0,84.16,86
48,49,40.533230,0.833980,38.031289,3723.0,11145.0,-7422.0,84.18,87
36,37,39.231269,0.855559,36.664592,2600.0,11650.0,-9050.0,88.08,88
57,58,38.855466,0.866387,36.256304,2533.0,11887.0,-9354.0,88.66,89


,id,Ranking Izquierda,skill iz,rank iz,Ranking Derecha,skill der,rank der,short_name,topic_name,name,ranking_derecha_sort,ranking_izquierda_sort,ranking_derecha_sort_skill,ranking_izquierda_sort_skill,ranking_derecha_sort_rank,ranking_izquierda_sort_rank
0,23,1,50.592856,1.07,75,39.381483,75.58,Nueva Constitución,Modernización del Estado,Nueva Constitución para Chile,75.000000,0.013333,0.778400,1.284686,70.635514,0.014157
1,55,2,48.919671,2.61,3,47.148776,6.02,Pensión Mínima,Pensiones,Pensión Mínima Igual o Mayor al Sueldo Mínimo,1.500000,0.666667,0.963800,1.037560,2.306513,0.433555
2,6,3,48.724362,3.12,46,42.499481,45.96,Agua,Economía,Desprivatización del Agua,15.333333,0.065217,0.872243,1.146470,14.730769,0.067885
3,85,4,47.978246,4.87,30,44.085301,32.53,Progresividad de Impuestos,Política Tributaria,Aumentar progresividad de impuestos (más impue...,7.500000,0.133333,0.918860,1.088305,6.679671,0.149708
4,37,5,47.970210,5.03,88,36.664592,88.08,Asamblea Constituyente,Modernización del Estado,Asamblea Constituyente,17.600000,0.056818,0.764320,1.308352,17.510934,0.057107
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,11,86,36.156062,85.80,81,38.742262,80.41,Educación RRSS,Educación,Educación sobre uso de Redes Sociales,0.941860,1.061728,1.071529,0.933246,0.937179,1.067031
86,80,87,35.666375,86.66,65,40.399510,66.50,Autopistas,Vivienda y Transporte,Más Autopistas Urbanas,0.747126,1.338462,1.132706,0.882842,0.767367,1.303158
87,49,88,34.507068,88.80,87,38.031289,84.18,Venta de Alcohol,Salud,"Restringir Venta de Alcohol en Delivery, Super...",0.988636,1.011494,1.102130,0.907334,0.947973,1.054882
88,52,89,34.503466,88.71,68,40.373771,66.73,Edad de Jubilación,Pensiones,Aumento Edad de Jubilación,0.764045,1.308824,1.170137,0.854601,0.752226,1.329387


,id,Ranking Izquierda,Ranking Derecha,skill iz,skill der,rank iz,rank der,ranking_izquierda_sort,ranking_derecha_sort
0,23,1,75,50.592856,39.381483,1.07,75.58,0.013333,75.000000
1,55,2,3,48.919671,47.148776,2.61,6.02,0.666667,1.500000
2,6,3,46,48.724362,42.499481,3.12,45.96,0.065217,15.333333
3,85,4,30,47.978246,44.085301,4.87,32.53,0.133333,7.500000
4,37,5,88,47.970210,36.664592,5.03,88.08,0.056818,17.600000
...,...,...,...,...,...,...,...,...,...
85,11,86,81,36.156062,38.742262,85.80,80.41,1.061728,0.941860
86,80,87,65,35.666375,40.399510,86.66,66.50,1.338462,0.747126
87,49,88,87,34.507068,38.031289,88.80,84.18,1.011494,0.988636
88,52,89,68,34.503466,40.373771,88.71,66.73,1.308824,0.764045
